In [1]:
import os
import pickle
import matplotlib.pyplot as plt
import torch
import numpy as np
from nnfabrik.builder import get_data, get_trainer

from model import stacked_core_full_gauss_readout
from trainer import standard_trainer
from sensorium.utility.scores import get_correlations

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from mog_vae.model import MoGVAE

device = "cuda:7"
torch.cuda.set_device(device)

In [2]:
basepath = "/srv/user/polina/sensorium/sensorium/notebooks/data/"

# as filenames, we'll select all 7 datasets
filenames = [
    os.path.join(basepath, file) for file in os.listdir(basepath) if ".zip" in file
]


dataset_fn = "sensorium.datasets.static_loaders"
dataset_config = {
    "paths": filenames,
    "normalize": True,
    "include_behavior": True,
    "include_eye_position": True,
    "batch_size": 128,
    "scale": 0.25,
}

dataloaders = get_data(dataset_fn, dataset_config)
data_keys = list(dataloaders['train'].keys())

In [3]:
model_config = {
    "pad_input": False,
    "stack": -1,
    "layers": 4,
    "input_kern": 9,
    "gamma_input": 6.3831,
    "gamma_readout": 0.0076,
    "hidden_kern": 7,
    "hidden_channels": 64,
    "depth_separable": True,
    "grid_mean_predictor": {
        "type": "cortex",
        "input_dimensions": 2,
        "hidden_layers": 1,
        "hidden_features": 30,
        "final_tanh": True,
    },
    "init_sigma": 0.1,
    "init_mu_range": 0.3,
    "gauss_type": "full",
    "shifter": True, 
    "autoencoder": None,
}

trainer_config = {
    'max_iter': 200,
    'verbose': False,
    'lr_decay_steps': 4,
    'avg_loss': False,
    'lr_init': 0.009,
    'device': device, 
}

In [4]:
model_seeds = [0, 1, 2]

In [5]:
from torch.nn import functional as F

def recon_loss_fn(x, x_rec):
    loss = (x - x_rec).pow(2).sum(-1)
    return loss.sum()

def log_normal(x, mu, var, eps=1e-8):
    var = var + eps
    return -0.5 * (np.log(2.0 * np.pi) + torch.log(var) + (x - mu).pow(2) / var).sum(dim=-1)

def gaussian_loss_fn(z, z_mu, z_var, z_mu_prior, z_var_prior):
    loss = log_normal(z, z_mu, z_var) - log_normal(z, z_mu_prior, z_var_prior)
    return loss.sum()

def entropy_loss_fn(logits, probs):
    log_q = F.log_softmax(logits, dim=-1)
    loss = -(probs * log_q).sum(dim=-1)
    return loss.sum()

In [6]:
def train_autoenc(autoencoder, features, w_rec, w_gauss, w_entr):
    optim = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=0.3, patience=100)

    epochs = int(1e15)
    min_lr = 1e-5

    for epoch in range(epochs):
        model_out = autoencoder(features, return_params=True)
        
        recon = model_out['x_rec']
        recon_loss = recon_loss_fn(features, recon)

        z = model_out['z']
        z_mu, z_var = model_out['mu'], model_out['var']
        z_mu_prior, z_var_prior = model_out['y_mu'], model_out['y_var']
        gaussian_loss = gaussian_loss_fn(z, z_mu, z_var, z_mu_prior, z_var_prior)

        logits, probs = model_out['logits'], model_out['probs']
        entropy_loss = entropy_loss_fn(logits, probs)

        loss = w_rec * recon_loss + w_gauss * gaussian_loss + w_entr * entropy_loss

        optim.zero_grad()
        loss.backward()
        optim.step()
        scheduler.step(loss.item())
        if epoch % 100 == 0:
            lr = optim.param_groups[0]['lr']
            if lr < min_lr:
                break

            print(recon_loss.item(), gaussian_loss.item(), entropy_loss.item())
            print(lr)

In [7]:
# Autoencoder configs for different latent dims that result in less than 1% performance drop
autoencoder_configs = [
    { 'latent_dim': 32, 'hidden_dims': 512, 'hidden_layers': 3, }, 
    { 'latent_dim': 16, 'hidden_dims': 512, 'hidden_layers': 5, }, 
    { 'latent_dim': 8, 'hidden_dims': 512, 'hidden_layers': 7, }, 
    { 'latent_dim': 4, 'hidden_dims': 1024, 'hidden_layers': 7, }, 
]

In [8]:
checkpoint = torch.load(f'checkpoints/base/model_weights{0}.pth')

# Extract features from all mice
features = []
for data_key in data_keys:
    features_single_mouse = checkpoint[f'readout.{data_key}._features']
    features_single_mouse = features_single_mouse.squeeze().permute(1, 0).detach()
    features.append(features_single_mouse)
features = torch.cat(features)

In [11]:
mog_vae = MoGVAE(64, 32, hidden_layers=3, hidden_dims=512, batch_norm=True, nonlinearity='GELU', num_components=5)
mog_vae.to(device)

MoGVAE(
  (inference): InferenceNet(
    (qyx_layers): ModuleList(
      (0): Linear(in_features=64, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=512, out_features=512, bias=True)
      (4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=512, out_features=512, bias=True)
      (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): GumbelSoftmax(
        (logits): Linear(in_features=512, out_features=5, bias=True)
      )
    )
    (qzyx_layers): ModuleList(
      (0): Linear(in_features=69, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_featur

In [12]:
train_autoenc(mog_vae, features, w_rec=100, w_gauss=1, w_entr=1)

3062517.5 1232758.875 75019.1875
0.001
142049.3125 2474708.5 12590.2001953125
0.001
107332.8515625 2744393.25 8941.796875
0.001
96143.2109375 2776797.25 7260.025390625
0.001
88798.59375 2808194.0 6290.7841796875
0.001
83299.609375 2830672.5 5597.564453125
0.001
78841.578125 2859160.0 4920.095703125
0.001
74304.484375 2870870.25 4437.4833984375
0.001
70421.046875 2895076.0 3913.93603515625
0.001
66763.09375 2923376.75 3598.732421875
0.001
62958.83203125 2953058.0 3099.4697265625
0.001
59636.27734375 2978047.5 2908.940673828125
0.001
56737.9453125 3000945.5 2722.4326171875
0.001
54014.76171875 3025597.5 2246.4814453125
0.001
51487.890625 3047275.75 2327.322265625
0.001
49370.7578125 3074582.5 2085.49853515625
0.001
47812.69921875 3090733.25 1979.573974609375
0.001
45802.8203125 3099770.25 1781.5511474609375
0.001
44563.578125 3122616.5 1497.9425048828125
0.001
42595.50390625 3135205.75 1369.849853515625
0.001
41168.171875 3151277.0 1314.934326171875
0.001
40311.703125 3156643.5 1272.5705

In [13]:
checkpoint = torch.load(f'checkpoints/base/model_weights{0}.pth')
model = stacked_core_full_gauss_readout(dataloaders, 0, **model_config)
model.load_state_dict(checkpoint)
model.eval()

validation_score = get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
)
print('base', validation_score)

mog_vae.eval()

for data_key in data_keys:
    model.readout[data_key].autoencoder = mog_vae

validation_score = get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
)
print(validation_score)

/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:74: UserWarning: Use of 'gamma_readout' is deprecated. Use 'feature_reg_weight' instead. If 'feature_reg_weight' is defined, 'gamma_readout' is ignored
  warnings.warn(
/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:95: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


base 0.38690987
0.37492678


In [27]:
ys = torch.eye(5)

mu, var = mog_vae.generative.pzy(ys)

In [45]:
mog_vae.to(device)

MoGVAE(
  (inference): InferenceNet(
    (qyx_layers): ModuleList(
      (0): Linear(in_features=64, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=512, out_features=512, bias=True)
      (4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=512, out_features=512, bias=True)
      (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): GumbelSoftmax(
        (logits): Linear(in_features=512, out_features=5, bias=True)
      )
    )
    (qzyx_layers): ModuleList(
      (0): Linear(in_features=69, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_featur

In [ ]:
# Cluster attribution numbers
logits, probs, y = mog_vae.inference.qyx(features, temperature=1., hard=False)
y = y.detach().cpu().numpy()
y = y.round()
y.sum(axis=0)

array([25627.,     0., 13621.,  2132., 13189.], dtype=float32)

In [ ]:
# Distance matrix of means
(mu[:, None, :] - mu[None, :, :]).pow(2).sum(dim=-1).sqrt()

tensor([[ 0.0000,  3.7800,  1.4089, 10.3518,  1.9073],
        [ 3.7800,  0.0000,  3.4168,  7.2538,  4.1340],
        [ 1.4089,  3.4168,  0.0000, 10.1503,  2.2470],
        [10.3518,  7.2538, 10.1503,  0.0000, 10.6109],
        [ 1.9073,  4.1340,  2.2470, 10.6109,  0.0000]],
       grad_fn=<SqrtBackward0>)

In [ ]:
# Train all autoencoders for each model

all_features = []

for model_seed in model_seeds:
    checkpoint = torch.load(f'checkpoints/base/model_weights{model_seed}.pth')

    # Extract features from all mice
    features = []
    for data_key in data_keys:
        features_single_mouse = checkpoint[f'readout.{data_key}._features']
        features_single_mouse = features_single_mouse.squeeze().permute(1, 0).detach()
        features.append(features_single_mouse)
    features = torch.cat(features)

    feature_dict = {
        'base': features.detach().cpu().numpy()
    }

    for autoencoder_config in autoencoder_configs:
        autoencoder = VAE(
            64, batch_norm=True, nonlinearity='GELU', **autoencoder_config
        )
        autoencoder.load_state_dict(
            torch.load(f'checkpoints/autoencoder/model_weights{model_seed}_{autoencoder_config['latent_dim']}.pth')
        )
        autoencoder.to(device)
        #train_autoenc(autoencoder, features)
        #torch.save(
        #    autoencoder.state_dict(), 
        #    f'checkpoints/autoencoder/model_weights{model_seed}_{autoencoder_config['latent_dim']}.pth'
        #)

        latent_features = autoencoder.encode(features)[0].detach().cpu().numpy()
        feature_dict[autoencoder_config['latent_dim']] = latent_features

    all_features.append(feature_dict)

    with open('all_features.pkl', 'wb') as f:
        pickle.dump(all_features, f)

NameError: name 'VAE' is not defined